# RPOE-X — GRPO Training on Google Colab

**OpenEnv Hackathon Submission** | [HF Space](https://bharavi-rpoe-x.hf.space) | [GitHub](https://huggingface.co/spaces/Bharavi/rpoe_x)

Train a Rotary Parking SRE agent using **GRPO** with HF TRL. The agent learns to route
cars across a 5-zone, 20-wheel rotary parking system in HITEC City, Hyderabad — importing all
training logic directly from the `rpoe_x` package.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://bharavi-rpoe-x.hf.space) (RPOE-X server) |
| Training | This Colab notebook (T4 / A100 GPU) |
| Model | `Qwen/Qwen2.5-0.5B-Instruct` + LoRA via PEFT |
| Framework | HF TRL v0.29 GRPO with vLLM backend |

## 1. Install Dependencies

Install the `rpoe_x` package (environment client + training utils) and TRL with vLLM backend.

In [ ]:
!pip install -q \
    "trl[vllm]==0.29.0" \
    "vllm>=0.11.0" \
    "peft" \
    "transformers" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "matplotlib" \
    "numpy" \
    "nest_asyncio"

# Install rpoe_x package (environment client + training utilities from HF Space)
!pip install -q "git+https://huggingface.co/spaces/Bharavi/rpoe_x"

import os
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")

## 2. Configuration

Set the HF Space URL, model, and training hyperparameters. Add `HF_TOKEN` to Colab Secrets
(key icon in the left sidebar).

In [ ]:
import os

# HuggingFace token
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except (ImportError, KeyError, ModuleNotFoundError):
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not set. Add it via Colab Secrets (left sidebar -> key icon).")

# Environment server (HF Space)
ENV_URL = "https://bharavi-rpoe-x.hf.space"

# Model
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
HUB_REPO = "Bharavi/rpoe-x-qwen-0.5b-grpo"

# Training hyperparameters
NUM_EPISODES    = 60   # episodes for dataset collection
MAX_STEPS       = 300  # GRPO training steps
NUM_GENERATIONS = 4    # completions generated per prompt
MAX_TURNS       = 50   # env steps per collection episode

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Max steps   : {MAX_STEPS}")

## 3. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset the environment, and run one action to confirm round-trip works.

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()  # must be applied before asyncio.run in Colab

from rpoe_x import ParkingEnv, ParkingAction

async def _smoke_test():
    async with ParkingEnv(base_url=ENV_URL) as env:
        reset_result = await env.reset()
        obs = reset_result.observation
        print("Connected!\n")
        print(f"Zone occupancy : {[round(x, 2) for x in obs.zone_occupancy]}")
        print(f"Queue lengths  : {obs.zone_queue_lengths}")
        print(f"Time of day    : {obs.time_of_day:.3f}")

        result = await env.step(ParkingAction(zone_id=2, wheel_id=0))
        print(f"\nSmoke test step (reward={result.reward:.3f})")
        print(f"  Zone occupancy: {[round(x, 2) for x in result.observation.zone_occupancy]}")

print(f"Connecting to {ENV_URL} ...")
asyncio.run(_smoke_test())
print("\nSmoke test passed. Environment is ready for training.")

## 4. Import Training Utilities from Package

Imports system prompts, formatters, parser, `reward_total` (pass-through), and
`build_rollout_func` from `rpoe_x.training.train`. The rollout function drives
live env interaction during training — no pre-collected dataset needed.

In [ ]:
import logging
import nest_asyncio
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

from rpoe_x.training.train import (
    ORCH_SYSTEM,
    ZONE_SYSTEM,
    format_orch_obs,
    format_zone_obs,
    parse_action,
    reward_total,
    build_rollout_func,
    plot_rewards,
)

nest_asyncio.apply()  # allow run_until_complete inside Colab event loop

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print(f"ORCH_SYSTEM (first 80 chars): {ORCH_SYSTEM[:80]}...")
print("Imported: reward_total, build_rollout_func, plot_rewards")

## 5. GRPO Training Setup

Define tokenizer, LoRA config, rollout function, and GRPOTrainer. Model loading is handled
internally by GRPOTrainer via vLLM — no manual `from_pretrained` needed.

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA config — GRPOTrainer applies this to the model on init
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Rollout function — each dataset entry triggers one live episode against the env
rollout_func = build_rollout_func(ENV_URL, tokenizer, max_turns=MAX_TURNS)
dataset = Dataset.from_dict({"prompt": ["Route the next car."] * NUM_EPISODES})

# Output directory
timestamp  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f"outputs/rpoe-x-grpo-{timestamp}"
Path(output_dir).mkdir(parents=True, exist_ok=True)
print(f"Output : {output_dir}")
print(f"Dataset: {len(dataset)} episodes (one live rollout each)")

In [ ]:
import sys, io

# vLLM colocate mode inherits stdout/stderr fds from the parent process.
# Colab's iostream raises UnsupportedOperation on fileno() — patch to fall back
# to the real fd numbers (1=stdout, 2=stderr) so vLLM worker spawn succeeds.
def _patch_fileno(stream, fallback_fd: int) -> None:
    original = stream.fileno
    def _fileno():
        try:
            return original()
        except io.UnsupportedOperation:
            return fallback_fd
    stream.fileno = _fileno

_patch_fileno(sys.stdout, 1)
_patch_fileno(sys.stderr, 2)

# GRPO Config
grpo_config = GRPOConfig(
    use_vllm                      = True,
    vllm_mode                     = "colocate",
    output_dir                    = output_dir,
    learning_rate                 = 5e-6,
    lr_scheduler_type             = "cosine",
    warmup_steps                  = 20,
    max_steps                     = MAX_STEPS,
    per_device_train_batch_size   = 2,
    gradient_accumulation_steps   = 4,
    generation_batch_size         = NUM_GENERATIONS,
    num_generations               = NUM_GENERATIONS,
    max_completion_length         = 128,
    temperature                   = 0.9,
    logging_steps                 = 10,
    save_strategy                 = "no",
    report_to                     = "none",
    seed                          = 42,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
)

# Pass MODEL_ID string — vLLM loads and serves the model for generation
trainer = GRPOTrainer(
    model            = MODEL_ID,
    processing_class = tokenizer,
    reward_funcs     = [reward_total],
    args             = grpo_config,
    train_dataset    = dataset,
    rollout_func     = rollout_func,
    peft_config      = peft_config,
)

print("GRPOTrainer initialized")
print(f"  Reward source : env.step() via rollout_func")
print(f"  Episodes      : {NUM_EPISODES} (live, using current model each step)")

## 6. Train!

Launch GRPO training. Each step: sample prompt from dataset → generate
`NUM_GENERATIONS` completions → score with reward functions → GRPO gradient update.

In [ ]:
print("Starting GRPO training ...")
print(f"  Model       : {MODEL_ID}")
print(f"  Max steps   : {MAX_STEPS}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Dataset     : {len(dataset):,} examples")
print()

trainer.train()
trainer.save_model(output_dir)

print(f"\nModel saved to {output_dir}")

## 7. Reward Curves

Visualise training progress using `plot_rewards` from the package.

In [ ]:
plot_rewards(trainer, save_path=f"{output_dir}/reward_curve.png")

## 8. Push to Hub (Optional)

Upload the trained LoRA adapter to HuggingFace Hub.

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ.get("HF_TOKEN"))
api.create_repo(repo_id=HUB_REPO, repo_type="model", exist_ok=True)

# Push adapter weights + tokenizer — no model reload needed
api.upload_folder(
    folder_path = output_dir,
    repo_id     = HUB_REPO,
    repo_type   = "model",
    commit_message = f"GRPO adapter — {timestamp}",
)

print(f"Pushed to https://huggingface.co/{HUB_REPO}")